In [1]:
## Import dependencies

import os, re, shutil, math, pickle, sys
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp
from scipy.special import softmax
from colabdesign import mk_afdesign_model, clear_mem
from colabdesign.mpnn import mk_mpnn_model
from colabdesign.af.alphafold.common import residue_constants, protein
from colabdesign.shared.protein import pdb_to_string
from colabdesign.af.loss import get_ptm, mask_loss, get_dgram_bins, _get_con_loss
from colabdesign.shared.utils import copy_dict

sys.path.append("/home/davidnannemann/git_repositories/BindCraft/")
from functions import *
from functions.biopython_utils import hotspot_residues, calculate_clash_score, calc_ss_percentage, calculate_percentages
from functions.pyrosetta_utils import pr_relax, align_pdbs
from functions.generic_utils import update_failures


ModuleNotFoundError: No module named 'colabdesign'

# Set up BindCraft pseudo-inputs

In [ ]:
starting_pdb = "inputs/PDL1.pdb"
design_name = "af2_model_exploration"

chain = "A"
length = 15

target_hotspot_residues = None
seed = 42

design_paths = {
    "Trajectory": "af2_model_exploration",
}

advanced_settings = {
    "af_params_dir":"",
    "use_multimer_design": True,
    "num_recycles_design": 1,
    "omit_AAs": "C",
    "rm_template_seq_design": False,
    "rm_template_sc_design": True,
}

# Generate AF2 model object

In [ ]:
model_pdb_path = os.path.join(design_paths["Trajectory"], design_name+".pdb")

# clear GPU memory for new trajectory
clear_mem()

In [ ]:
af_model = mk_afdesign_model(protocol="binder", debug=True, data_dir=advanced_settings["af_params_dir"],
                                use_multimer=advanced_settings["use_multimer_design"], num_recycles=advanced_settings["num_recycles_design"],
                                best_metric='loss')

In [ ]:
af_model.prep_inputs(pdb_filename=starting_pdb, chain=chain, binder_len=length, hotspot=target_hotspot_residues, seed=seed, rm_aa=advanced_settings["omit_AAs"],
                        rm_target_seq=advanced_settings["rm_template_seq_design"], rm_target_sc=advanced_settings["rm_template_sc_design"])

# Explore inputs within the `af_model` object

The inputs to AF2 include definitions for different parts of the model.
- the portions of the model to utilize as target or design as binder
   - with the "binder" functionality, the `prep_model` function sets up supervised losses for the target and allows full re-design of the binder
   - held one-hot encoded in the model inputs as `entidy_id`
- target template coordinates
   - the template coordinates for the binder are included. Zero coordinates are initiated for the binder.
   - held in the model inputs as `batch[all_atom_positions]`
- the target sequence
   - controlled with the BindCraft advanced option `rm_template_seq`
   - held in the model inputs as `rm_template_seq`
   - the input sequence for the target and the undefined sequence (as zeros) for the binder
   - removing the template sequence from the target turns off backbone constraints, allowing for greater structural diversity in the output
- whether or not to provide the side chain coordinates
   - controlled with the BindCraft advanced option `rm_template_sc`
   - held in the model inputs as `rm_template_sc`
   - removal further increases structural diversity of the target during design

For advanced functionality, values can be edited once the model is set up.


In [ ]:
af_model._inputs.keys()

dict_keys(['aatype', 'target_feat', 'msa_feat', 'seq_mask', 'msa_mask', 'msa_row_mask', 'atom14_atom_exists', 'atom37_atom_exists', 'residx_atom14_to_atom37', 'residx_atom37_to_atom14', 'residue_index', 'extra_deletion_value', 'extra_has_deletion', 'extra_msa', 'extra_msa_mask', 'extra_msa_row_mask', 'template_aatype', 'template_all_atom_mask', 'template_all_atom_positions', 'template_mask', 'template_pseudo_beta', 'template_pseudo_beta_mask', 'asym_id', 'sym_id', 'entity_id', 'all_atom_positions', 'batch', 'rm_template', 'rm_template_seq', 'rm_template_sc', 'bias'])

In [ ]:
print_input_keys = ["residue_index", "entity_id", "batch","rm_template","rm_template_seq","rm_template_sc"]

# print the model inputs
for key in af_model._inputs.keys():
    #if not key in print_input_keys: continue
    print("--------------------------------")
    if key == "batch":
        for batch_key in af_model._inputs["batch"].keys():
            print("batch",batch_key, af_model._inputs["batch"][batch_key].shape)
            print(af_model._inputs["batch"][batch_key])
    else:
        print(key, af_model._inputs[key].shape)
        print(af_model._inputs[key])

--------------------------------
aatype (130,)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
--------------------------------
target_feat (130, 20)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
--------------------------------
msa_feat (1, 130, 49)
[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]]
--------------------------------
seq_mask (130,)
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.

# Food for Thought

What other inputs might be useful to edit? Could you
- add an MSA?
- Include a template for the binder?